In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd

df = pd.read_csv(r'Dataset\comparison_outputs\Movies-Dataset-no-missing-2000-2026-ML-Ready-encoded.csv')
# --- 1. KEYWORDS PROCESSING ---
# Replace hyphens with spaces so CountVectorizer can read them as separate phrases
df['keywords_clean'] = df['keywords'].astype(str).str.replace('-', ' ')

# Get keyword counts
df['num_keywords'] = df['keywords_clean'].apply(lambda x: len(x.split()) if x != 'nan' else 0)

# Extract only the Top 25 Keywords
kw_vectorizer = CountVectorizer(max_features=25, stop_words='english')
kw_matrix = kw_vectorizer.fit_transform(df['keywords_clean'])
kw_df = pd.DataFrame(kw_matrix.toarray(), columns=[f"kw_{c}" for c in kw_vectorizer.get_feature_names_out()], index=df.index)

# --- 2. TAGLINE & OVERVIEW PROCESSING ---
# Fill missing text with empty strings
df['tagline'] = df['tagline'].fillna('')
df['overview'] = df['overview'].fillna('')

# Meta-features: Word counts
df['overview_word_count'] = df['overview'].apply(lambda x: len(x.split()))
df['tagline_word_count'] = df['tagline'].apply(lambda x: len(x.split()))

# Combine them into one narrative block
df['combined_text'] = df['tagline'] + " " + df['overview']

# TF-IDF to extract important words, max 500 words to save memory
tfidf = TfidfVectorizer(stop_words='english', max_features=500)
text_tfidf_matrix = tfidf.fit_transform(df['combined_text'])

# Compress those 500 words down into 5 "Theme" dimensions using SVD (PCA for text)
svd = TruncatedSVD(n_components=5, random_state=42)
text_svd_matrix = svd.fit_transform(text_tfidf_matrix)

# Create a DataFrame for the new text components
svd_df = pd.DataFrame(text_svd_matrix, columns=[f"text_theme_{i+1}" for i in range(5)], index=df.index)

# --- 3. MERGE EVERYTHING BACK ---
df = pd.concat([df, kw_df, svd_df], axis=1)

# Drop the original massive text columns so they don't crash your pipeline
df = df.drop(columns=['keywords', 'keywords_clean', 'tagline', 'overview', 'combined_text'])

C:\Users\kabhi\AppData\Local\Temp\ipykernel_8664\1168008522.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['num_keywords'] = df['keywords_clean'].apply(lambda x: len(x.split()) if x != 'nan' else 0)
C:\Users\kabhi\AppData\Local\Temp\ipykernel_8664\1168008522.py:24: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['overview_word_count'] = df['overview'].apply(lambda x: len(x.split()))
C:\Users\kabhi\AppData\Local\Temp\ipykernel_8664\1168008522.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually th

In [3]:
from pathlib import Path
dataset_path = r'Dataset\comparison_outputs\Movies-Dataset-no-missing-2000-2026-ML-Ready-encoded.csv'
DATA_PATH = Path(dataset_path)
OUTPUT_PATH = DATA_PATH.parent / f"{DATA_PATH.stem}-encoded.csv"

df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved encoded dataset to: {OUTPUT_PATH}")

Saved encoded dataset to: Dataset\comparison_outputs\Movies-Dataset-no-missing-2000-2026-ML-Ready-encoded-encoded.csv
